In [74]:
# In Silver layer, we are reading the parquet file data from the Bronze layer into dataframes to do the transformations.

In [9]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("silver_layer_cleaning").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/16 08:05:03 WARN Utils: Your hostname, MacBook-Air-105.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.90 instead (on interface en0)
26/03/16 08:05:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 08:05:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/16 08:05:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [10]:
# Reading the parquet data from bronze layer into Dataframes

customers_bronze = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_customers_dataset")
orders_bronze = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_orders_dataset")
order_items_bronze = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_order_items_dataset")
payments_bronze = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_order_payments_dataset")


In [11]:
# Transforming Customers Table

In [12]:
customers_bronze.show()

+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+---------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state| ingestion_timestamp|  source_system|
+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+---------------+
|f2a1d75b74d9ec748...|15ee900ec703c9a10...|                   68590|             jacunda|            PA|2026-02-28 08:51:...|olist_ecommerce|
|f15272fe9d0e2ae32...|11e74a9cbe1158d1c...|                   15056|sao jose do rio p...|            SP|2026-02-28 08:51:...|olist_ecommerce|
|7324ecb0ff143f561...|c6be127fa6e30c6f7...|                   13302|                 itu|            SP|2026-02-28 08:51:...|olist_ecommerce|
|7accf3d920f47c07f...|a7f1a6dc9ba06844b...|                   45638|             coaraci|            BA|2026-02-28 08:51:...|olist_ecommerce|
|3680a

In [13]:
customers_bronze.describe().show()

26/03/16 08:05:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 5:=======================================>                   (2 + 1) / 3]

+-------+--------------------+--------------------+------------------------+-------------------+--------------+---------------+
|summary|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|  source_system|
+-------+--------------------+--------------------+------------------------+-------------------+--------------+---------------+
|  count|               99441|               99441|                   99441|              99441|         99441|          99441|
|   mean|                NULL|                NULL|       35137.47458291851|               NULL|          NULL|           NULL|
| stddev|                NULL|                NULL|       29797.93899620613|               NULL|          NULL|           NULL|
|    min|00012a2ce6f8dcda2...|0000366f3b9a7992b...|                    1003|abadia dos dourados|            AC|olist_ecommerce|
|    max|ffffe8b65bbe3087b...|ffffd2657e2aad290...|                   99990|             zortea|        

In [14]:
customers_bronze.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)



In [15]:
# Checking for duplicates

customers_bronze.selectExpr("count(*) AS total_count", "count(DISTINCT customer_id) AS distinct_customer_count").show()

[Stage 8:===================>                                       (1 + 2) / 3]

+-----------+-----------------------+
|total_count|distinct_customer_count|
+-----------+-----------------------+
|      99441|                  99441|
+-----------+-----------------------+



In [16]:
# Trim and Capitalize columns

from pyspark.sql.functions import trim, col, initcap, upper
customers_silver = customers_bronze.select(trim(col("customer_id")).alias("customer_id"),
                                          trim(col("customer_unique_id")).alias("customer_unique_id"),
                                           col("customer_zip_code_prefix"),
                                           initcap(trim(col("customer_city"))).alias("customer_city"),
                                           upper(trim(col("customer_state"))).alias("customer_state"),
                                           )

In [17]:
customers_silver.show()

[Stage 14:>                                                         (0 + 1) / 1]

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|f2a1d75b74d9ec748...|15ee900ec703c9a10...|                   68590|             Jacunda|            PA|
|f15272fe9d0e2ae32...|11e74a9cbe1158d1c...|                   15056|Sao Jose Do Rio P...|            SP|
|7324ecb0ff143f561...|c6be127fa6e30c6f7...|                   13302|                 Itu|            SP|
|7accf3d920f47c07f...|a7f1a6dc9ba06844b...|                   45638|             Coaraci|            BA|
|3680a273ddb333253...|6cbfcc29787035834...|                   29700|            Colatina|            ES|
|f18edbd308dd8784f...|ef6e328edc28262f4...|                   18055|            Sorocaba|            SP|
|d7492c2b1d4bcba6c...|d681f09d6d055742a...|            

In [18]:
# Checking if State has any values not equal to 2 letters

from pyspark.sql.functions import length
customers_silver.filter(length("customer_state") != 2).show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
+-----------+------------------+------------------------+-------------+--------------+



In [19]:
#Transforming Orders Table

In [20]:
orders_bronze.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| ingestion_timestamp|  source_system|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+
|6ec1bea8cbcef0a1b...|e6b5e20566e5c72cb...|   delivered|     2018-08-07 05:42:46|2018-08-07 05:50:08|         2018-08-13 09:24:00|          2018-08-27 15:28:22|          2018-09-18 00:00:00|2026-02-28 08:51:...|olist_ecommerce|
|441972a5bbd51a104...|b79fa9dfed0c3d624...|   delivered|     2018-08-23 22:37:44|2018-08

In [21]:
orders_bronze.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)



In [22]:
orders_bronze.describe().show()

[Stage 18:==============>                                           (1 + 3) / 4]

+-------+--------------------+--------------------+------------+---------------+
|summary|            order_id|         customer_id|order_status|  source_system|
+-------+--------------------+--------------------+------------+---------------+
|  count|               99441|               99441|       99441|          99441|
|   mean|                NULL|                NULL|        NULL|           NULL|
| stddev|                NULL|                NULL|        NULL|           NULL|
|    min|00010242fe8c5a6d1...|00012a2ce6f8dcda2...|    approved|olist_ecommerce|
|    max|fffe41c64501cc87c...|ffffe8b65bbe3087b...| unavailable|olist_ecommerce|
+-------+--------------------+--------------------+------------+---------------+



In [23]:
# Checking for null primary keys(order_id)

orders_bronze.filter(col("order_id").isNull()).count()

0

In [24]:
# Checking for duplicate order_id

from pyspark.sql.functions import countDistinct
orders_bronze.select(countDistinct("order_id").alias("distinct_order_id")).show()

[Stage 24:==============>                                           (1 + 3) / 4]

+-----------------+
|distinct_order_id|
+-----------------+
|            99441|
+-----------------+



In [25]:
# Checking if multiple orders have same customer repeated

orders_bronze.select(countDistinct("customer_id").alias("distinct_customer_id")).show()

+--------------------+
|distinct_customer_id|
+--------------------+
|               99441|
+--------------------+



In [26]:
# Checking distinct order and customer id combination

orders_bronze.select(countDistinct("order_id", "customer_id").alias("distinct_order_customer_id")).show()

+--------------------------+
|distinct_order_customer_id|
+--------------------------+
|                     99441|
+--------------------------+



In [27]:
# Trim and Standardize string columns

from pyspark.sql.functions import trim, lower, col

orders_silver = orders_bronze.select(trim(col("order_id")).alias("order_id"),
                     trim(col("customer_id")).alias("customer_id"),
                     lower(trim(col("order_status"))).alias("order_status"),
                     col("order_purchase_timestamp"),
                     col("order_approved_at"),
                     col("order_delivered_carrier_date"),
                     col("order_delivered_customer_date"),
                     col("order_estimated_delivery_date"),
                    )
orders_silver.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|6ec1bea8cbcef0a1b...|e6b5e20566e5c72cb...|   delivered|     2018-08-07 05:42:46|2018-08-07 05:50:08|         2018-08-13 09:24:00|          2018-08-27 15:28:22|          2018-09-18 00:00:00|
|441972a5bbd51a104...|b79fa9dfed0c3d624...|   delivered|     2018-08-23 22:37:44|2018-08-24 00:35:13|         2018-08-24 13:19:00|          2018-08-29 18:03:26|          2018-09-04 00:00:00|
|fbecebecbe32df9dc...|7f9f88f14a8f0dc73...|  

In [28]:
orders_silver.select("order_status").distinct().show()

+------------+
|order_status|
+------------+
|     shipped|
|    canceled|
|    approved|
|    invoiced|
|   delivered|
| unavailable|
|  processing|
|     created|
+------------+



In [29]:
# Validating Timestamp logic

orders_silver.filter(col("order_purchase_timestamp") > col("order_approved_at")).count()


0

In [30]:

orders_silver.filter(col("order_approved_at") > col("order_delivered_customer_date")).count()


61

In [31]:
# Deriving a new column for invalid orders as Order Approval Date is greater than Order Delivered Date

from pyspark.sql.functions import when
orders_silver = orders_silver.withColumn("approval_delivery_valid", when(col("order_approved_at") < col("order_delivered_customer_date"),
                "valid")
                .otherwise("invalid"))
orders_silver.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|approval_delivery_valid|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+
|6ec1bea8cbcef0a1b...|e6b5e20566e5c72cb...|   delivered|     2018-08-07 05:42:46|2018-08-07 05:50:08|         2018-08-13 09:24:00|          2018-08-27 15:28:22|          2018-09-18 00:00:00|                  valid|
|441972a5bbd51a104...|b79fa9dfed0c3d624...|   delivered|     2018-08-23 22:37:44|2018-08-24 00:35:13|         2018-08-24 13:19:00|          

In [32]:
# Deriving a new column for Total no. of days to deliver

from pyspark.sql.functions import datediff

orders_silver = orders_silver.withColumn("delivery_days", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp").cast("date"))
                                        )
orders_silver.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+-------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|approval_delivery_valid|delivery_days|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+-------------+
|6ec1bea8cbcef0a1b...|e6b5e20566e5c72cb...|   delivered|     2018-08-07 05:42:46|2018-08-07 05:50:08|         2018-08-13 09:24:00|          2018-08-27 15:28:22|          2018-09-18 00:00:00|                  valid|           20|
|441972a5bbd51a104...|b79fa9dfed0c3d624...|   delivered|     2018-08-23 22:37:44|201

In [33]:
# Joining Orders table with Customers Table

In [34]:
joined_orders_customers = orders_silver.join(customers_silver, on = "customer_id", how = "left")
joined_orders_customers.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+-------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|approval_delivery_valid|delivery_days|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+-------------+--------------------+------------------------+--------------------+--------------+
|e6b5e20566e5c72cb...|6ec1bea8cbcef0a1b...|   delivered|     2018-0

In [35]:
joined_orders_customers.cache()

DataFrame[customer_id: string, order_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, approval_delivery_valid: string, delivery_days: int, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string]

In [36]:
joined_orders_customers.count()

99441

In [37]:
joined_orders_customers.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

[Stage 61:==============>                                           (1 + 3) / 4]

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [38]:
# Transforming Order Items Table

In [39]:
order_items_bronze.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)



In [40]:
order_items_bronze.describe().show()

[Stage 66:==============>                                           (1 + 3) / 4]

+-------+--------------------+------------------+--------------------+--------------------+------------------+------------------+---------------+
|summary|            order_id|     order_item_id|          product_id|           seller_id|             price|     freight_value|  source_system|
+-------+--------------------+------------------+--------------------+--------------------+------------------+------------------+---------------+
|  count|              112650|            112650|              112650|              112650|            112650|            112650|         112650|
|   mean|                NULL|1.1978339991122948|                NULL|                NULL|120.65373901464987|19.990319928982817|           NULL|
| stddev|                NULL|0.7051240313951752|                NULL|                NULL|183.63392805025975|15.806405412297076|           NULL|
|    min|00010242fe8c5a6d1...|                 1|00066f42aeeb9f300...|0015a82c2db000af6...|              0.85|              

In [41]:
order_items_bronze.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+---------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value| ingestion_timestamp|  source_system|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+---------------+
|a292ffb436f475f98...|            1|5f21301936c11698d...|fe2032dab1a61af87...|2017-06-29 12:23:42| 149.9|        34.85|2026-02-28 08:52:...|olist_ecommerce|
|a29361cea3ac5ecb5...|            1|5a3320037d5922a77...|b6d44737c04332870...|2018-03-01 15:55:27|  79.0|        24.36|2026-02-28 08:52:...|olist_ecommerce|
|a29361cea3ac5ecb5...|            2|7a6aebc4c1205818e...|8bd0f31cf0a614c65...|2018-03-01 15:55:27|199.99|        26.23|2026-02-28 08:52:...|olist_ecommerce|
|a2940f1b67316eac4...|            1|d0fe4295267f15cca...|0

In [42]:
# Checking for nulls in order_id & product_id

In [43]:
order_items_bronze.filter(col("product_id").isNull()).count()

0

In [44]:
order_items_bronze.filter(col("order_id").isNull()).count()

0

In [45]:
# Trimming Columns

In [46]:
order_items_silver = order_items_bronze.select(trim(col("order_id")).alias("order_id"), col("order_item_id"), 
                                               trim(col("product_id")).alias("product_id"), 
                                               trim(col("seller_id")).alias("seller_id"), col("shipping_limit_date"), 
                                               col("price"),
                                               col("freight_value")
)
order_items_silver.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|a292ffb436f475f98...|            1|5f21301936c11698d...|fe2032dab1a61af87...|2017-06-29 12:23:42| 149.9|        34.85|
|a29361cea3ac5ecb5...|            1|5a3320037d5922a77...|b6d44737c04332870...|2018-03-01 15:55:27|  79.0|        24.36|
|a29361cea3ac5ecb5...|            2|7a6aebc4c1205818e...|8bd0f31cf0a614c65...|2018-03-01 15:55:27|199.99|        26.23|
|a2940f1b67316eac4...|            1|d0fe4295267f15cca...|0db783cfcd3b73998...|2017-06-30 16:55:11|  49.9|        11.85|
|a2941bd059891a933...|            1|0aabfb375647d9738...|37515688008a7a40a...|2017-11-28 03:06:59|  24.5|        11.85|
|a294a9571b11828c8...|            1|386e

In [47]:
# Deriving new columns and rounding the integer

In [48]:
from pyspark.sql.functions import col, round
order_items_silver = order_items_silver.withColumn("total_item_value", round((col("price") + col("freight_value")),2))\
                                        .withColumn("freight_ratio", round(col("freight_value")/col("price"), 2))\
                                        .withColumn("is_high_freight", when(col("freight_ratio") > 0.5, "Yes"). otherwise("No"))

order_items_silver.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+----------------+-------------+---------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|total_item_value|freight_ratio|is_high_freight|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+----------------+-------------+---------------+
|a292ffb436f475f98...|            1|5f21301936c11698d...|fe2032dab1a61af87...|2017-06-29 12:23:42| 149.9|        34.85|          184.75|         0.23|             No|
|a29361cea3ac5ecb5...|            1|5a3320037d5922a77...|b6d44737c04332870...|2018-03-01 15:55:27|  79.0|        24.36|          103.36|         0.31|             No|
|a29361cea3ac5ecb5...|            2|7a6aebc4c1205818e...|8bd0f31cf0a614c65...|2018-03-01 15:55:27|199.99|        26.23|          226.22|         0.13|             No

In [49]:
order_items_silver.count()

112650

In [50]:
joined_orders_customers.unpersist()

DataFrame[customer_id: string, order_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, approval_delivery_valid: string, delivery_days: int, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string]

In [51]:
# Join Orders & Customers with Order items and to derive a new column

In [52]:
joined_orders_customers_order_items = (
    joined_orders_customers.join(order_items_silver,on = "order_id", how = "inner")
    .withColumn("shipping_limit_days", datediff(col("shipping_limit_date"), col("order_purchase_timestamp")))
)
joined_orders_customers_order_items.show()


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+-------------+--------------------+------------------------+--------------------+--------------+-------------+--------------------+--------------------+-------------------+-----+-------------+----------------+-------------+---------------+-------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|approval_delivery_valid|delivery_days|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|total_item_value|freight_ratio|is_high_freight|shipping_limit_days|
+--------------------+--------------------+---------

In [53]:
joined_orders_customers_order_items.count()

112650

In [54]:
joined_orders_customers_order_items.cache()

DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, approval_delivery_valid: string, delivery_days: int, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double, total_item_value: double, freight_ratio: double, is_high_freight: string, shipping_limit_days: int]

In [55]:
# Count total items per Order and sort by highest count

In [56]:
joined_orders_customers_order_items.groupBy("order_id").count().orderBy(col("count"), ascending = False).show()

[Stage 91:==============>                                           (1 + 3) / 4]

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|8272b63d03f5f79c5...|   21|
|1b15974a0141d54e3...|   20|
|ab14fdcfbe524636d...|   20|
|428a2f660dc84138d...|   15|
|9ef13efd6949e4573...|   15|
|73c8ab38f07dc9438...|   14|
|9bdc4d4c71aa1de46...|   14|
|37ee401157a3a0b28...|   13|
|c05d6a79e55da72ca...|   12|
|af822dacd6f5cff73...|   12|
|637617b3ffe9e2f7a...|   12|
|2c2a19b5703863c90...|   12|
|3a213fcdfe7d98be7...|   12|
|71dab1155600756af...|   11|
|7f2c22c54cbae5509...|   11|
|6c355e2913545fa6f...|   11|
|5a3b1c29a49756e75...|   11|
|ca3625898fbd48669...|   10|
|9f5054bd9a3c71702...|   10|
|9aec4e1ae90b23c7b...|   10|
+--------------------+-----+
only showing top 20 rows


In [57]:
# Checking for any duplicate (order_id, order_item_id) pair

In [58]:
dup_order_orderitem = joined_orders_customers_order_items.groupBy("order_id", "order_item_id").count().filter("count > 1")
dup_order_orderitem.show()

[Stage 95:==============>                                           (1 + 3) / 4]

+--------+-------------+-----+
|order_id|order_item_id|count|
+--------+-------------+-----+
+--------+-------------+-----+



In [59]:
# Transforming Payments Table

In [60]:
payments_bronze.describe().show()

[Stage 100:============================>                            (1 + 1) / 2]

+-------+--------------------+------------------+------------+--------------------+------------------+---------------+
|summary|            order_id|payment_sequential|payment_type|payment_installments|     payment_value|  source_system|
+-------+--------------------+------------------+------------+--------------------+------------------+---------------+
|  count|              103886|            103886|      103886|              103886|            103886|         103886|
|   mean|                NULL|1.0926785129853878|        NULL|   2.853348863176944|154.10038041698792|           NULL|
| stddev|                NULL|0.7065837791949958|        NULL|   2.687050673856492|  217.494063864724|           NULL|
|    min|00010242fe8c5a6d1...|                 1|      boleto|                   0|               0.0|olist_ecommerce|
|    max|fffe41c64501cc87c...|                29|     voucher|                  24|          13664.08|olist_ecommerce|
+-------+--------------------+------------------

In [61]:
payments_bronze.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)



In [62]:
payments_bronze.show()

+--------------------+------------------+------------+--------------------+-------------+--------------------+---------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value| ingestion_timestamp|  source_system|
+--------------------+------------------+------------+--------------------+-------------+--------------------+---------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|2026-02-28 08:52:...|olist_ecommerce|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|2026-02-28 08:52:...|olist_ecommerce|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|2026-02-28 08:52:...|olist_ecommerce|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|2026-02-28 08:52:...|olist_ecommerce|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|2026-02-28 08:52:...|o

In [88]:
payments_bronze.filter(col("order_id").isNull()).show()

+--------+------------------+------------+--------------------+-------------+-------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|ingestion_timestamp|source_system|
+--------+------------------+------------+--------------------+-------------+-------------------+-------------+
+--------+------------------+------------+--------------------+-------------+-------------------+-------------+



In [91]:
payments_bronze.select("payment_type").distinct().show()

+------------+
|payment_type|
+------------+
|      boleto|
| not_defined|
| credit_card|
|     voucher|
|  debit_card|
+------------+



In [92]:
payments_bronze.groupBy("payment_type").count().show()

+------------+-----+
|payment_type|count|
+------------+-----+
|      boleto|19784|
| not_defined|    3|
| credit_card|76795|
|     voucher| 5775|
|  debit_card| 1529|
+------------+-----+



In [104]:
payments_silver = payments_bronze.select(col("order_id"), col("payment_sequential"),\
                                         trim(col("payment_type")).alias("payment_type"),\
                                         col("payment_installments"), col("payment_value")
                                        )
payments_silver.show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [105]:
payments_silver.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [107]:
from pyspark.sql.functions import sum, round, collect_set, col
payments_silver.groupBy("order_id").agg(
                                    round(sum(("payment_value")),2).alias("total_value"),
                                    collect_set("payment_type").alias("payment_types")
                                        ).orderBy(col("total_value"),ascending = False).show()

[Stage 159:============================>                            (1 + 1) / 2]

+--------------------+-----------+-------------+
|            order_id|total_value|payment_types|
+--------------------+-----------+-------------+
|03caa2c082116e1d3...|   13664.08|[credit_card]|
|736e1922ae60d0d6a...|    7274.88|     [boleto]|
|0812eb902a67711a1...|    6929.31|[credit_card]|
|fefacc66af859508b...|    6922.21|     [boleto]|
|f5136e38d1a14a4db...|    6726.66|     [boleto]|
|2cc9089445046817a...|    6081.54|     [boleto]|
|a96610ab360d42a2e...|    4950.34|[credit_card]|
|b4c4b76c642808cbe...|    4809.44|[credit_card]|
|199af31afc78c699f...|    4764.34|[credit_card]|
|8dbc85d1447242f3b...|    4681.78|[credit_card]|
|426a9742b533fc6fe...|    4513.32|[credit_card]|
|d2f270487125ddc41...|     4445.5| [debit_card]|
|80dfedb6d17bf2353...|    4194.76|[credit_card]|
|68101694e5c5dc733...|    4175.26|[credit_card]|
|b239ca7cd485940b3...|    4163.51| [debit_card]|
|9a3966c23190dbdba...|    4042.74|[credit_card]|
|9de73f3e6157169ad...|    4034.44|[credit_card]|
|86c4eab1571921a6a..

In [111]:
payments_silver.count()

103886

In [112]:
payments_silver = payments_silver.dropDuplicates()
payments_silver.count()

103886

In [113]:
payments_silver.groupBy("order_id").count().orderBy("count", ascending = False).show()

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|fa65dad1b0e818e3c...|   29|
|ccf804e764ed5650c...|   26|
|285c2e15bebd4ac83...|   22|
|895ab968e7bb0d565...|   21|
|fedcd9f7ccdc8cba3...|   19|
|ee9ca989fc93ba09a...|   19|
|21577126c19bf11a0...|   15|
|4bfcba9e084f46c8e...|   15|
|3c58bffb70dcf45f1...|   14|
|4689b1816de42507a...|   14|
|cf101c3abd3c061ca...|   13|
|4fb76fa13b108a0d0...|   13|
|73df5d6adbeea12c8...|   13|
|6d58638e32674bebe...|   12|
|465c2e1bee4561cb3...|   12|
|c6492b842ac190db8...|   12|
|67d83bd36ec2c7fb5...|   12|
|1a611328643ae1114...|   12|
|1d9a9731b9c10fc9c...|   12|
|d744783ed2ace06ca...|   12|
+--------------------+-----+
only showing top 20 rows


In [114]:
payments_silver.head()

Row(order_id='b81ef226f3fe1789b1e8b2acac839d17', payment_sequential=1, payment_type='credit_card', payment_installments=8, payment_value=99.33)

In [115]:
payments_silver.filter(col("order_id").like("fedcd9f7ccdc8cba3%")).orderBy(col("payment_sequential")).show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|fedcd9f7ccdc8cba3...|                 1| credit_card|                   1|         1.67|
|fedcd9f7ccdc8cba3...|                 2|     voucher|                   1|         7.76|
|fedcd9f7ccdc8cba3...|                 3|     voucher|                   1|        26.94|
|fedcd9f7ccdc8cba3...|                 4|     voucher|                   1|        10.33|
|fedcd9f7ccdc8cba3...|                 5|     voucher|                   1|         9.76|
|fedcd9f7ccdc8cba3...|                 6|     voucher|                   1|          8.6|
|fedcd9f7ccdc8cba3...|                 7|     voucher|                   1|        11.78|
|fedcd9f7ccdc8cba3...|                 8|     voucher|                   1|        31.43|
|fedcd9f7c

In [ ]:
#Joining Payments Table

In [116]:
joined_orders_customers_payments = joined_orders_customers.join(payments_silver, on = "order_id", how = "inner")
joined_orders_customers_payments.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+-------------+--------------------+------------------------+--------------------+--------------+------------------+------------+--------------------+-------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|approval_delivery_valid|delivery_days|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+-------------+----------------

In [117]:
joined_orders_customers_payments.count()

103886

In [118]:
dup_counts = joined_orders_customers_payments.groupBy("order_id", "customer_id").count().filter(col("count") >1)
dup_counts.show()

[Stage 212:==============>                                          (1 + 3) / 4]

+--------------------+--------------------+-----+
|            order_id|         customer_id|count|
+--------------------+--------------------+-----+
|f50b6f864012d10ca...|2d231c527c22c03b8...|    2|
|58b3c61516dc5ed61...|c13b43440b0555633...|    4|
|0c65eb5a1415db7a2...|eda4c5f48f975dff9...|    3|
|87a9c87504af54b4d...|889987ae94247fa15...|    2|
|f63a31c3349b87273...|bf04d8d8923728c5b...|    2|
|7d77cbd8b988f81a2...|75aa0f7b76847ca2b...|    2|
|52a637177b682c31c...|f70ad1bcdbade5ca6...|    2|
|b6d3f75a92e19fb13...|e22bf63c919b667e3...|    2|
|ccf804e764ed5650c...|92cd3ec6e2d643d4e...|   26|
|3a98a62b1abeae3d8...|4e839bbfb46074711...|    2|
|692e004e31546b86a...|9e77a8bef5c8d0d18...|    3|
|919baca007d9525b6...|144d6b2679193370e...|    2|
|8029473fa5eb709b7...|4d6e3cd7620b52c39...|    2|
|d192eacb35dd2f0aa...|e5867bb6374f11c01...|    3|
|b579612f258cd7fd3...|7213b48aa052d444b...|    2|
|66408ac1fd2db1448...|4bc5733793dc86f89...|    2|
|678c60c6679ddcf17...|730325254d70acdb1...|    2|


In [119]:
joined_orders_customers_payments.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

[Stage 219:==============>                                          (1 + 3) / 4]

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|a1e28dc56f8cf4e56...|    2|
|a9b929204be626d61...|    2|
|1826d2a2eb6ba6e3e...|    2|
|6386b8aabb7f032b7...|    2|
|8dd9758206f8d9c23...|    2|
|5ded9a59e8920225f...|    2|
|fee9afa24ed743a26...|    2|
|f63a31c3349b87273...|    2|
|35ab20ce8b706d545...|    2|
|0fa927b252421189a...|    4|
|421b4ffa3b5162cfd...|    2|
|749990ae992d496bb...|    2|
|97abc984480b7b063...|    2|
|7f344d881ecc07068...|    2|
|234ea4333eec78d8e...|    2|
|6064862631581009b...|    4|
|b5b143bb15757cfbb...|    2|
|c991b8532e7abe41b...|    2|
|66408ac1fd2db1448...|    2|
|55205d82f3b72a235...|    2|
+--------------------+-----+
only showing top 20 rows


In [120]:
# This statement initially gave that difference in 1 record, which was happening due to left join orders_customers and payments table
# Because there was 1 row in orders_customers table which had no payment, so when i did inner join, that difference in 1 row wiped out

joined_orders_customers_payments.select("order_id") \
    .exceptAll(payments_silver.select("order_id")) \
    .show()

[Stage 228:===================================>                     (5 + 3) / 8]

+--------+
|order_id|
+--------+
+--------+



In [121]:
customers_silver.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/silver/customers")
orders_silver.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/silver/orders")
order_items_silver.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/silver/order_items")
payments_silver.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/silver/payments")
